# Part 3 — Sizing the Recommendations

For each recommendation: state the assumption, then size the one-year change in **GMV**, **profit (contribution)**
and **credit loss**.

**Method**
- **Volume = 2025** (a full year of originations).
- **Rates = seasoned loans** (originated by Jun-2025), so margins and loss rates aren't distorted by young loans.
- `loans_clean.csv` is a sample of Affirm's volume, so read the $ figures as **relative** sizes.

Input: `loans_clean.csv` from Part 1.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np

loans_clean = pd.read_csv("loans_clean.csv")
loans_clean["origination_date"] = pd.to_datetime(loans_clean["origination_date"])

# Unit economics per loan (same as Part 2)
is_split = loans_clean["product_type"] == "split_pay"
term_years = np.where(is_split, 1.5, loans_clean["loan_term_months"]) / 12
loans_clean["revenue"]      = loans_clean["mdr_fees"] + loans_clean["interest_collected"]
loans_clean["funding_cost"] = loans_clean["loan_amount"] * loans_clean["cost_of_funds_bps"] / 10000 * term_years * 0.5
loans_clean["contribution"] = loans_clean["revenue"] - loans_clean["credit_loss"] - loans_clean["funding_cost"]

loans_2025 = loans_clean[loans_clean["origination_date"].dt.year == 2025]
seasoned   = loans_clean[loans_clean["origination_date"] <= "2025-06-30"]

In [ ]:
# Steady-state margin and loss rate (% of GMV) by vertical, from seasoned loans
by_vertical = seasoned.groupby("vertical")[["loan_amount", "contribution", "credit_loss"]].sum()
by_vertical["margin"]    = by_vertical["contribution"] / by_vertical["loan_amount"]
by_vertical["loss_rate"] = by_vertical["credit_loss"] / by_vertical["loan_amount"]
margin, loss_rate = by_vertical["margin"], by_vertical["loss_rate"]

# Baseline year: 2025 volume x steady-state rates
gmv_2025 = loans_2025.groupby("vertical")["loan_amount"].sum()
base_gmv    = gmv_2025.sum()
base_profit = (gmv_2025 * margin).sum()
base_loss   = (gmv_2025 * loss_rate).sum()

print(f"Baseline: GMV ${base_gmv:,.0f} | profit ${base_profit:,.0f} | loss rate {base_loss / base_gmv:.2%}")
by_vertical[["margin", "loss_rate"]].round(4)

## 1. Grow healthcare & home furnishing through merchant acquisition
**Assumption:** a new merchant performs like the average existing merchant in its vertical.

In [ ]:
# What does one merchant bring in a year? (2025)
per_merchant = loans_2025.groupby(["vertical", "merchant_id"]).agg(consumers=("user_id", "nunique"),
                                                                  gmv=("loan_amount", "sum")).reset_index()
per_merchant["profit"] = per_merchant["gmv"] * per_merchant["vertical"].map(margin)

targets = per_merchant[per_merchant["vertical"].isin(["healthcare", "home_furnishing"])]
targets.round(0)

In [ ]:
avg_merchant = targets.groupby("vertical").agg(merchants=("merchant_id", "count"),
                                               consumers=("consumers", "mean"),
                                               gmv=("gmv", "mean"),
                                               profit=("profit", "mean"))
avg_merchant.round(0)

In [ ]:
# New merchants needed for +10% / +20% GMV in each vertical
for growth in [0.10, 0.20]:
    print(f"GMV +{growth:.0%}:")
    for v, row in avg_merchant.iterrows():
        extra_gmv = gmv_2025[v] * growth
        print(f"  {v:16s} +${extra_gmv:,.0f} GMV = {extra_gmv / row['gmv']:.1f} new merchants,"
              f" ~{extra_gmv / row['gmv'] * row['consumers']:.0f} new consumers")

In [ ]:
def grow_verticals(growth):
    extra_gmv = gmv_2025[["healthcare", "home_furnishing"]] * growth
    return {"gmv": extra_gmv.sum(),
            "profit": (extra_gmv * margin).sum(),
            "loss": (extra_gmv * loss_rate).sum()}

rec1 = {"conservative": grow_verticals(0.10), "ambitious": grow_verticals(0.20)}
pd.DataFrame(rec1).round(0)

An average healthcare merchant brings ~57 consumers and ~$104K GMV a year (~$10K profit); home furnishing ~122
consumers and ~$121K GMV (~$10K profit). With only 3–4 merchants per vertical, **one new merchant adds ~25–35%** —
so +10% needs ~0.3–0.4 of an average merchant and +20% ~0.6–0.8. Both verticals lose less than the portfolio average,
so this growth also lowers the overall loss rate.

## 2. Turn first-time customers into repeat customers
Two levers (Part 2, 1.5):
- **(a)** Raise the repeat rate in big-ticket verticals (healthcare, travel, home furnishing, auto parts: ~20% today).
- **(b)** Steer second loans to interest-bearing instead of Split Pay.

In [ ]:
repeat_loans = seasoned[~seasoned["is_new_to_affirm"]]
big_ticket = ["healthcare", "travel", "home_furnishing", "auto_parts"]

# (a) Average repeat loan in big-ticket verticals, and how many first-time customers they had in 2025
avg_repeat = repeat_loans[repeat_loans["vertical"].isin(big_ticket)][["loan_amount", "contribution", "credit_loss"]].mean()
new_customers = (loans_2025["is_new_to_affirm"] & loans_2025["vertical"].isin(big_ticket)).sum()
print("First-time big-ticket customers in 2025:", new_customers)
avg_repeat.round(1)

In [ ]:
# (b) Average repeat loan by product: what changes if a Split Pay repeat loan becomes interest-bearing?
avg_by_product = repeat_loans.groupby("product_type")[["loan_amount", "contribution", "credit_loss"]].mean()
switch = avg_by_product.loc["installment_ib"] - avg_by_product.loc["split_pay"]
n_repeat_2025 = (~loans_2025["is_new_to_affirm"]).sum()
print("Repeat loans in 2025:", n_repeat_2025)
avg_by_product.round(1)

In [ ]:
def repeat_levers(repeat_pts, switch_share):
    extra_repeat = new_customers * repeat_pts     # (a) more customers come back
    switched = n_repeat_2025 * switch_share       # (b) repeat loans moved Split Pay -> interest-bearing
    return {"gmv":    extra_repeat * avg_repeat["loan_amount"]  + switched * switch["loan_amount"],
            "profit": extra_repeat * avg_repeat["contribution"] + switched * switch["contribution"],
            "loss":   extra_repeat * avg_repeat["credit_loss"]  + switched * switch["credit_loss"]}

rec2 = {"conservative": repeat_levers(0.05, 0.05), "ambitious": repeat_levers(0.10, 0.10)}
pd.DataFrame(rec2).round(0)

Conservative: +5 pts repeat rate and 5% of repeat loans switched; ambitious: +10 pts and 10%.
An average big-ticket repeat loan earns ~$126; switching a repeat loan from Split Pay to interest-bearing adds ~$49.

## 3. Reduce exposure to the riskiest segment: travel, FICO < 640
**Assumption:** the existing risk model's approval cut-off and credit limits are tightened for this segment so that
its loan volume falls by **25% (conservative) or 50% (ambitious)**; profit and loss shrink in proportion.

In [ ]:
segment_2025 = loans_2025[(loans_2025["vertical"] == "travel") & (loans_2025["fico_score"] < 640)]
segment_seasoned = seasoned[(seasoned["vertical"] == "travel") & (seasoned["fico_score"] < 640)]

seg_gmv    = segment_2025["loan_amount"].sum()
seg_margin = segment_seasoned["contribution"].sum() / segment_seasoned["loan_amount"].sum()
seg_loss   = segment_seasoned["credit_loss"].sum() / segment_seasoned["loan_amount"].sum()
print(f"Travel FICO<640: 2025 GMV ${seg_gmv:,.0f} ({seg_gmv / base_gmv:.1%} of total) | margin {seg_margin:.1%} | loss rate {seg_loss:.1%}")

def cut_segment(cut):
    gmv = -seg_gmv * cut
    return {"gmv": gmv, "profit": gmv * seg_margin, "loss": gmv * seg_loss}

rec3 = {"conservative": cut_segment(0.25), "ambitious": cut_segment(0.50)}
pd.DataFrame(rec3).round(0)

**Trade-off:** this segment is still profitable on the base loss assumption, so cutting it gives up a little profit
in exchange for removing the portfolio's highest-loss pocket (6.4% loss rate vs 2.3% overall).

## 4. Combined impact (one year)

In [ ]:
summary = {}
for scenario in ["conservative", "ambitious"]:
    parts = pd.DataFrame({"1. Grow healthcare + home furnishing": rec1[scenario],
                          "2. Repeat customers":                  rec2[scenario],
                          "3. Tighten travel FICO<640":           rec3[scenario]}).T
    total = parts.sum()
    summary[scenario] = {
        "GMV change":        total["gmv"] / base_gmv,
        "profit change":     total["profit"] / base_profit,
        "loss rate before":  base_loss / base_gmv,
        "loss rate after":   (base_loss + total["loss"]) / (base_gmv + total["gmv"]),
    }
    print(scenario); print(parts.round(0), "\n")

pd.DataFrame(summary).round(4)

## Summary
| | Conservative | Ambitious |
|---|---|---|
| Healthcare + home furnishing GMV | +10% | +20% |
| Big-ticket repeat rate / repeat loans switched to interest-bearing | +5 pts / 5% | +10 pts / 10% |
| Travel FICO<640 volume | −25% | −50% |
| **GMV** | **+3.9%** | **+7.7%** |
| **Profit (contribution)** | **+7.4%** | **+14.8%** |
| **Credit loss rate** | **2.32% → 2.21%** | **2.32% → 2.10%** |

Profit grows faster than GMV and the loss rate falls: growth goes to low-loss verticals and repeat customers, and
exposure is cut where risk is concentrated. (0% APR repricing is discussed in the slides, not sized here.)